# Phase 3 — Rule-based arm

Runs the deterministic `smart_spatial_system` accessibility pipeline on
`data/processed/` (phase 2) and demonstrates the rule-based arm's
baseline for `paper/comparison_metric.md`: **PAR = 1.0** and
**Rank Stability (ρ) = 1.0**, "by construction" - not a result to
compute, but worth actually running/asserting here rather than only
asserting in prose. No network, no LLM key needed
(see `paper/PLAN.md`'s phase table).

**Update:** this notebook's extraction logic was wrong on the first real
run (`outputs["sites_ranked"]` is an opaque `VectorOut`, not something
`pd.DataFrame()` can eat directly) - fixed below using
`outputs["sites_report"].table["rows"]` instead, confirmed against the
actual saved output. The rest (imports, `build_accessibility_query_spec`,
`build_accessibility_initial_inputs`, the DAG execution call) ran clean
the first time. If something *else* still errors, that's still possible
- this has only been run once.


## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd

from orchestrator.capability_registry import CapabilityRegistry
from orchestrator.planning.dag_executor import DagExecutor
from orchestrator.planning.planner import DeterministicPlanner
from smart_spatial_system.application.services.query_execution.accessibility_query_spec import (
    AmenitySpec,
    build_accessibility_initial_inputs,
    build_accessibility_query_spec,
)

PROCESSED = Path("../data/processed")
RESULTS = Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)

# smart_spatial_system's own DEFAULT_TARGET_CRS is EPSG:3857 (Web Mercator)
# - fine globally but inflates distances ~1.5x at Vienna's latitude (see
# CLAUDE.md / phase 2 notebook). Its own Vienna example overrides this to
# EPSG:31256 (MGI / Austria GK East); we do the same, for the same reason.
TARGET_CRS = "EPSG:31256"
RAW_QUERY = "Rank Vienna's 23 municipal districts by accessibility to metro, schools, and parks"
N_RUNS = 5  # rule-based arm has no source of variation; this just proves it


## Load phase 2 outputs

`sites` and each amenity layer are passed as plain GeoJSON dicts (not GeoDataFrames) - confirmed from `build_accessibility_initial_inputs`'s signature.

In [2]:
def load_geojson(name):
    with open(PROCESSED / f"{name}.geojson") as f:
        return json.load(f)

sites_geojson = load_geojson("districts")
amenity_geojson = {
    "metro": load_geojson("metro"),
    "schools": load_geojson("schools"),
    "parks": load_geojson("parks"),
}

amenity_specs_raw = json.loads((PROCESSED / "amenity_specs.json").read_text())
AMENITIES = [AmenitySpec(**spec) for spec in amenity_specs_raw]

print(f"sites: {len(sites_geojson['features'])} districts")
for ref, gj in amenity_geojson.items():
    print(f"amenity '{ref}': {len(gj['features'])} features")
print()
for a in AMENITIES:
    print(a)


sites: 23 districts
amenity 'metro': 109 features
amenity 'schools': 211 features
amenity 'parks': 1063 features

AmenitySpec(ref='metro', distance_field='distance_to_metro_m', max_distance_m=800.0, weight=3.0, label='Metro (U-Bahn) station')
AmenitySpec(ref='schools', distance_field='distance_to_school_m', max_distance_m=1000.0, weight=2.0, label='School')
AmenitySpec(ref='parks', distance_field='distance_to_park_m', max_distance_m=1500.0, weight=1.0, label='Park')


## Build the QuerySpec (the rule-based plan itself)

In [3]:
query_spec = build_accessibility_query_spec(
    RAW_QUERY,
    AMENITIES,
    target_crs=TARGET_CRS,
)

# This operation-name sequence is exactly what paper/comparison_metric.md's
# Plan Agreement Rate (Layer 1) compares across runs (stripped of node ids
# and parameter values - just names + wiring). Print it now so it's visible
# before we assert anything about it below.
op_sequence = [op.name if hasattr(op, "name") else str(op) for op in query_spec.operations]
print(f"{len(op_sequence)} operations:")
for name in op_sequence:
    print(" ", name)


10 operations:
  OperationSpec(op='crs_transform', inputs={'vector': 'sites'}, params={'source_crs': 'EPSG:4326', 'target_crs': 'EPSG:31256'}, output='sites_metric')
  OperationSpec(op='crs_transform', inputs={'vector': 'metro'}, params={'source_crs': 'EPSG:4326', 'target_crs': 'EPSG:31256'}, output='metro_metric')
  OperationSpec(op='nearest_neighbor', inputs={'source': 'sites_metric', 'target': 'metro_metric'}, params={'k': 1, 'distance_field': 'distance_to_metro_m'}, output='sites_with_metro')
  OperationSpec(op='crs_transform', inputs={'vector': 'schools'}, params={'source_crs': 'EPSG:4326', 'target_crs': 'EPSG:31256'}, output='schools_metric')
  OperationSpec(op='nearest_neighbor', inputs={'source': 'sites_with_metro', 'target': 'schools_metric'}, params={'k': 1, 'distance_field': 'distance_to_school_m'}, output='sites_with_schools')
  OperationSpec(op='crs_transform', inputs={'vector': 'parks'}, params={'source_crs': 'EPSG:4326', 'target_crs': 'EPSG:31256'}, output='parks_metric'

## Execute the DAG

Defensive/exploratory on purpose - first run, real data. Print the result's actual shape before assuming a schema.

In [4]:
initial_inputs = build_accessibility_initial_inputs(
    sites=sites_geojson,
    amenity_layers=amenity_geojson,
)

registry = CapabilityRegistry.from_plugin_modules(tolerant=True)
plan = DeterministicPlanner().build(query_spec)
result = DagExecutor(lambda name: registry.resolve(name).callable).execute(
    plan, initial_inputs=initial_inputs,
)

print("success:", result.success)
print("result attrs:", [a for a in dir(result) if not a.startswith("_")])
if hasattr(result, "outputs"):
    print("output keys:", list(result.outputs.keys()))


success: True
result attrs: ['error', 'output_nodes', 'outputs', 'structured_error', 'success', 'trace']
output keys: ['sites_metric', 'metro_metric', 'schools_metric', 'parks_metric', 'sites_with_metro', 'sites_with_schools', 'sites_with_parks', 'sites_scored', 'sites_ranked', 'sites_report']


In [5]:
# Confirmed on a real run (2026-09-12): outputs['sites_ranked'] is a
# geochat_sdk.types.vector.VectorOut (an opaque wrapper, not directly
# useful here) and outputs['sites_report'] is a plugins.report_builder.
# ReportOut dataclass with plain-dict .meta / .summary / .table fields -
# .table['rows'] is a list of per-district dicts with exactly the columns
# we need (rank, name, accessibility_score, distance_to_*_m). Printing
# .summary here instead of the full table (which was ~370KB of repr on
# the run that confirmed this - 23 rows is small, but a naive print of
# the wrapping objects is not).
report = result.outputs["sites_report"]
print("type(sites_report):", type(report))
print("meta:", report.meta)
print("summary:", report.summary)
print("table columns:", [c["field"] for c in report.table["columns"]])
print("table row count:", len(report.table["rows"]))


type(sites_report): <class 'plugins.report_builder.ReportOut'>
meta: {'title': 'Accessibility Analysis Report', 'language': 'en', 'format': 'pdf', 'generated_at': '2026-09-12T22:12:45.253292+00:00', 'feature_count': 23, 'score_field': 'accessibility_score', 'rank_field': 'rank', 'plugin': 'report_builder'}
summary: {'total_count': 23, 'language': 'en', 'top_score': 100.0, 'min_score': 99.16, 'max_score': 100.0, 'avg_score': 99.93, 'median_score': 100.0, 'top_name': 'Innere Stadt', 'top_rank': 1, 'top_score_value': 100.0}
table columns: ['rank', 'name', 'accessibility_score', 'distance_to_metro_m', 'distance_to_school_m', 'distance_to_park_m']
table row count: 23


### Extracting the ranking

In [6]:
def extract_ranking(result, score_field="accessibility_score", rank_field="rank"):
    report = result.outputs["sites_report"]
    df = pd.DataFrame(report.table["rows"])
    missing = [c for c in (score_field, rank_field) if c not in df.columns]
    if missing:
        raise KeyError(f"expected columns {missing} not in report table columns {list(df.columns)}")
    return df.sort_values(rank_field).reset_index(drop=True)

ranking = extract_ranking(result)
ranking


,rank,name,accessibility_score,distance_to_metro_m,distance_to_school_m,distance_to_park_m
0,1,Innere Stadt,100.0,0.0,0.0,0.0
1,2,Leopoldstadt,100.0,0.0,0.0,0.0
2,3,Landstraße,100.0,0.0,0.0,0.0
3,4,Wieden,100.0,0.0,0.0,0.0
4,5,Margareten,100.0,0.0,0.0,0.0
5,6,Mariahilf,100.0,0.0,0.0,0.0
6,7,Neubau,100.0,0.0,0.0,0.0
7,8,Josefstadt,100.0,0.0,0.0,0.0
8,9,Alsergrund,100.0,0.0,0.0,0.0
9,10,Favoriten,100.0,0.0,0.0,0.0


**Observation - lots of ties at the top.** On the confirmed run, 11 of 23
districts tied at `accessibility_score = 100.0` with all three distances
at `0.0`. That's not a bug: `nearest_neighbor` measures distance from
each district *polygon* to the nearest amenity point, so any district
that simply *contains* at least one metro station, one school, and one
park gets `0.0` on all three and saturates the score - which, for a
dense city core, is a lot of districts. Worth a sentence in the paper's
discussion (a centroid-to-point or population-weighted distance would
differentiate the top of the ranking more; polygon-to-point is a
defensible but coarser choice) - not something to silently change here.


## Determinism check (Layer 1 + Layer 3 baseline)

Rebuild the QuerySpec and re-execute the whole pipeline from scratch, `N_RUNS` times. There's no randomness anywhere in this arm, so this should trivially hold - but assert it rather than assume it, exactly as phase 4 will have to for the LLM arm.

In [7]:
from scipy.stats import spearmanr

op_sequences = []
rankings = []

for i in range(N_RUNS):
    qs = build_accessibility_query_spec(RAW_QUERY, AMENITIES, target_crs=TARGET_CRS)
    op_sequences.append(tuple(op.name if hasattr(op, "name") else str(op) for op in qs.operations))

    plan_i = DeterministicPlanner().build(qs)
    result_i = DagExecutor(lambda name: registry.resolve(name).callable).execute(
        plan_i, initial_inputs=initial_inputs,
    )
    assert result_i.success, f"run {i}: DAG execution failed"
    rankings.append(extract_ranking(result_i))

# Layer 1 - Plan Agreement Rate
reference_seq = op_sequences[0]
par = sum(seq == reference_seq for seq in op_sequences) / N_RUNS
print(f"PAR = {par}")
assert par == 1.0, "rule-based arm should be byte-identical across runs by construction"

# Layer 3 - Rank Stability: pairwise Spearman rho between each pair of
# runs' rank *vectors*, aligned by district name (report.table's rows
# carry "name", not the district admin "ref" - name is unique per
# district and stable across runs, so it's a fine join key here). Spearman
# needs each run's rank for the *same fixed order* of districts, not the
# name lists themselves.
site_names = sorted(rankings[0]["name"])  # fixed common ordering

def rank_vector(df):
    rank_by_name = dict(zip(df["name"], df["rank"]))
    return [rank_by_name[n] for n in site_names]

rhos = []
for i in range(N_RUNS):
    for j in range(i + 1, N_RUNS):
        rho, _ = spearmanr(rank_vector(rankings[i]), rank_vector(rankings[j]))
        rhos.append(rho)

rank_stability = float(pd.Series(rhos).mean()) if rhos else 1.0  # numpy float64 -> plain float, for json.dumps below
print(f"Rank Stability (mean rho) = {rank_stability}")
assert all(r == 1.0 for r in rhos), "rule-based arm's ranking should be identical across runs by construction"


PAR = 1.0
Rank Stability (mean rho) = 1.0


## Save canonical rule-based results for phase 5

All `N_RUNS` are identical by construction, so persist one canonical run - `notebooks/04_comparison_metric.ipynb` (phase 5) needs this to compute ρ between each LLM run and this reference ranking.

In [8]:
ranking.to_csv(RESULTS / "rule_based_ranking.csv", index=False)

(RESULTS / "rule_based_query_spec.json").write_text(json.dumps({
    "raw_query": RAW_QUERY,
    "target_crs": TARGET_CRS,
    "amenities": amenity_specs_raw,
    "operation_sequence": list(op_sequence),
    "par": par,
    "rank_stability": rank_stability,
    "n_runs_checked": N_RUNS,
}, indent=2))

print("wrote", RESULTS / "rule_based_ranking.csv")
print("wrote", RESULTS / "rule_based_query_spec.json")


wrote ../results/rule_based_ranking.csv
wrote ../results/rule_based_query_spec.json


## Final checks

In [9]:
check = pd.read_csv(RESULTS / "rule_based_ranking.csv")
meta = json.loads((RESULTS / "rule_based_query_spec.json").read_text())

assert len(check) == 23, f"expected 23 ranked districts, got {len(check)}"
assert sorted(check["rank"].tolist()) == list(range(1, 24)), "rank column should be a permutation of 1..23"
assert meta["par"] == 1.0 and meta["rank_stability"] == 1.0

print("all checks passed")
print()
print(check.head(23))


all checks passed

    rank                  name  accessibility_score  distance_to_metro_m  \
0      1          Innere Stadt                100.0                  0.0   
1      2          Leopoldstadt                100.0                  0.0   
2      3            Landstraße                100.0                  0.0   
3      4                Wieden                100.0                  0.0   
4      5            Margareten                100.0                  0.0   
5      6             Mariahilf                100.0                  0.0   
6      7                Neubau                100.0                  0.0   
7      8            Josefstadt                100.0                  0.0   
8      9            Alsergrund                100.0                  0.0   
9     10             Favoriten                100.0                  0.0   
10    11             Simmering                100.0                  0.0   
11    12              Meidling                100.0                  

## Next: phase 4

The LLM arm (`notebooks/03_llm_arm.ipynb`) needs an `.env` with an LLM key
(see `.env.example`) and a real network connection - unlike phases 1-3,
this one costs real API calls, so `results/llm_runs/` (once produced)
gets committed rather than re-run casually. It will build the *same*
`AMENITY_SPECS` (from `data/processed/amenity_specs.json`, not
retyped) into whatever `LLMQuerySpecGenerator` expects, run it `N` times
(`paper/PLAN.md`'s open decision: N=20 as a starting point), and save
each run's raw `QuerySpec` + ranking - exactly the same shapes this
notebook just produced for the rule-based arm's single canonical run -
so phase 5 can compute Plan Agreement Rate / parametric variance / Rank
Stability uniformly across both arms.
